# 01. F1 Data Exploration & Ingestion

This notebook demonstrates how we load Grand Prix timing data via FastF1, clean it, apply fuel-corrections, and fit piecewise tire degradation curves with a cliff.

In [ ]:
import os
import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit

sns.set_theme(style="whitegrid")

## 1. Load Silverstone 2022 Race

In [ ]:
fastf1.Cache.enable_cache('../data/raw')
session = fastf1.get_session(2022, 'Silverstone', 'R')
session.load(laps=True, telemetry=False, weather=True)

laps = session.laps.copy()
laps["LapTime_s"] = laps["LapTime"].dt.total_seconds()
laps = laps.dropna(subset=["Compound", "TyreLife", "LapTime_s"])
laps["Compound"] = laps["Compound"].str.upper()
print(f"Loaded {len(laps)} laps from {session.event['EventName']} {session.event['EventDate'].year}")

## 2. Plot Raw Lap Times vs Tire Age

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=laps, x="TyreLife", y="LapTime_s", hue="Compound", alpha=0.6)
plt.title("Raw Lap Times vs Tire Age - Silverstone 2022")
plt.xlabel("Tire Age (Laps)")
plt.ylabel("Lap Time (seconds)")
plt.show()

## 3. Apply Fuel Correction

Subtract linear fuel weight penalty (~1.9 kg/lap, ~0.033 s/kg) to isolate tire degradation.

In [ ]:
total_laps = 52
laps["laps_remaining"] = total_laps - laps["LapNumber"]
laps["fuel_kg"] = laps["laps_remaining"] * 1.9
laps["fuel_effect"] = laps["fuel_kg"] * 0.033
laps["LapTime_corrected"] = laps["LapTime_s"] - laps["fuel_effect"]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=laps, x="TyreLife", y="LapTime_corrected", hue="Compound", alpha=0.6)
plt.title("Fuel-Corrected Lap Times vs Tire Age")
plt.xlabel("Tire Age (Laps)")
plt.ylabel("Corrected Lap Time (seconds)")
plt.show()

## 4. Normalize per Driver-Stint and Fit Piecewise Degradation Curves

In [ ]:
# Normalize stint to remove driver/car pace offset
clean_laps = laps[(laps["TrackStatus"] == "1") & (laps["IsAccurate"] == True)].copy()
normalized_list = []
for (driver, stint), group in clean_laps.groupby(["Driver", "Stint"]):
    if len(group) < 3:
        continue
    baseline = group["LapTime_corrected"].quantile(0.05)
    group["deg"] = group["LapTime_corrected"] - baseline
    normalized_list.append(group)
df_norm = pd.concat(normalized_list, ignore_index=True)

def piecewise_deg(age, base_deg, cliff_age, cliff_sev):
    gradual = base_deg * age
    cliff = cliff_sev * np.maximum(0, age - cliff_age) ** 2
    return gradual + cliff

# Fit for MEDIUM compound
med_laps = df_norm[df_norm["Compound"] == "MEDIUM"]
age_median = med_laps.groupby("TyreLife")["deg"].median().reset_index()

popt, _ = curve_fit(piecewise_deg, age_median["TyreLife"].values, age_median["deg"].values, p0=[0.05, 20.0, 0.2], bounds=([0, 5, 0], [0.5, 45, 2.0]))
base_deg, cliff_age, cliff_sev = popt
print(f"Fitted Params - Base Deg: {base_deg:.4f} s/lap, Cliff Age: {cliff_age:.1f} laps, Cliff Severity: {cliff_sev:.4f}")

# Plot Fit
plt.figure(figsize=(10, 6))
plt.scatter(age_median["TyreLife"], age_median["deg"], label="Median Deg (Data)", color="orange")
x_fit = np.linspace(1, 35, 100)
y_fit = piecewise_deg(x_fit, base_deg, cliff_age, cliff_sev)
plt.plot(x_fit, y_fit, label="Fitted Piecewise Curve", color="blue", lw=2)
plt.axvline(cliff_age, color="red", linestyle="--", label=f"Cliff Age ({cliff_age:.1f})")
plt.title("MEDIUM Tire Degradation Piecewise Fit")
plt.xlabel("Tire Age (Laps)")
plt.ylabel("Degradation Penalty (seconds)")
plt.legend()
plt.show()

## 5. Safety Car Laps Frequency

In [ ]:
laps["sc_active"] = laps["TrackStatus"].astype(str).str.contains("4")
sc_by_lap = laps.groupby("LapNumber")["sc_active"].mean().reset_index()

plt.figure(figsize=(10, 5))
plt.bar(sc_by_lap["LapNumber"], sc_by_lap["sc_active"], color="red", alpha=0.7)
plt.title("Safety Car Active Probability by Lap Number")
plt.xlabel("Lap Number")
plt.ylabel("Active Rate (Fraction of Drivers)")
plt.show()